# Preparación de datos

**Conjunto de datos:** Dataset 2 - Lugares de emisiones

**Nombre de archivo:** emission_permits_anom_2.json

## 0. Inicialización

Importaciones

In [53]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import re
import json
from tabulate import tabulate

Visualización de tablas y gráficas

In [54]:
sns.set_style("darkgrid")

def print_table(df):
    print(tabulate(df, headers='keys', tablefmt='simple_outline'))

Lectura y muestra del archivo

In [55]:
# 1. Cargar el JSON
with open('../data/original/emission_permits_anom_2.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 2. Extraer features
features = data['features']

# 3. Construir DataFrame con properties y coordenadas
df = pd.DataFrame([
    {
        **feature['properties'],
        'Latitud': feature['geometry']['coordinates'][1],
        'Longitud': feature['geometry']['coordinates'][0]
    }
    for feature in features
])

df.head()

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Cuenca,Latitud,Longitud
0,73640.0,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,CENTRO,None,Otros,Horno,Río Bogotá,4.703418,-74.226561
1,73788.0,Seguimiento y Control,Ubate,Cundinamarca,LENGUAZAQUE,Resguardo,None,Carbón,Caldera Horno,Río Suárez,5.318407,-73.704281
2,74314.0,Seguimiento y Control,Sabana Occidente,Cundinamarca,MADRID,LA PUNTA,None,ACPM,Caldera Horno,Río Bogotá,4.800462,-74.210355
3,75972.0,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,Balsillas,None,Fuel Oil No.8,Planta de Asfalto,Río Bogotá,4.678797,-74.284112
4,78824.0,Seguimiento y Control,Sabana Occidente,Cundinamarca,FUNZA,El Hato,None,Carbón,Caldera Horno,Río Bogotá,4.699590,-74.193752


**1.** Transformar IDExpediente a integer

In [56]:
df['IDExpediente'] = df['IDExpediente'].astype("Int64")

**2.** Eliminar columnas innecesarias

In [57]:
df = df.drop(columns=['Cuenca'])

**3.** Eliminar duplicados

In [58]:
df = df.drop_duplicates()

**4.** Manejar la capitalización

In [59]:
df["Vereda"] = df["Vereda"].str.strip().str.upper()
df["TipoFuenteEmision"] = df["TipoFuenteEmision"].str.strip().str.capitalize()
df.head()

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Latitud,Longitud
0,73640,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,CENTRO,None,Otros,Horno,4.703418,-74.226561
1,73788,Seguimiento y Control,Ubate,Cundinamarca,LENGUAZAQUE,RESGUARDO,None,Carbón,Caldera horno,5.318407,-73.704281
2,74314,Seguimiento y Control,Sabana Occidente,Cundinamarca,MADRID,LA PUNTA,None,ACPM,Caldera horno,4.800462,-74.210355
3,75972,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,BALSILLAS,None,Fuel Oil No.8,Planta de asfalto,4.678797,-74.284112
4,78824,Seguimiento y Control,Sabana Occidente,Cundinamarca,FUNZA,EL HATO,None,Carbón,Caldera horno,4.699590,-74.193752


**5.** Mejorar la presentación de los (sin definir)

In [60]:
df["TipoCombustible"] = df["TipoCombustible"].replace("(sin definir)", "Sin definir")
df["TipoFuenteEmision"] = df["TipoFuenteEmision"].replace("(sin definir)", "Sin definir")
df[(df["TipoCombustible"] == "Sin definir") | (df["TipoFuenteEmision"] == "Sin definir")]

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Latitud,Longitud
42,138622,Seguimiento y Control,Sabana Centro,Cundinamarca,NEMOCON,PATIO BONITO,None,Sin definir,Sin definir,5.118908,-73.895435
43,139320,Seguimiento y Control,Ubate,Cundinamarca,TAUSA,RASGATÁ,None,Sin definir,Horno,5.190613,-73.879585
61,150306,Seguimiento y Control,Sabana Centro,Cundinamarca,NEMOCON,PATIO BONITO,None,Sin definir,Sin definir,5.125421,-73.904398
100,171196,Seguimiento y Control,Bogotá y Municipio de la Calera,Distrito Capital,LOCALIDAD DE CIUDAD BOLIVAR,MOCHUELO BAJO,None,Sin definir,Sin definir,4.506914,-74.150135
102,171204,Seguimiento y Control,Bogotá y Municipio de la Calera,Distrito Capital,LOCALIDAD DE CIUDAD BOLIVAR,MOCHUELO BAJO,None,Sin definir,Sin definir,4.516030,-74.149127
...,...,...,...,...,...,...,...,...,...,...,...
539,291446,Sancionatorio,Chiquinquira,Boyacá,RAQUIRA,CASCO URBANO,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,12.629099,-39.893628
540,291712,Sancionatorio,Sumapaz,Cundinamarca,ARBELAEZ,SAN ROQUE,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,4.274013,-74.446186
541,292030,Sancionatorio,Ubate,Cundinamarca,CUCUNUBA,PUEBLO VIEJO,Por emisiones atmosféricas sin permiso o no cu...,Sin definir,Sin definir,5.228254,-73.817856
542,292322,Sancionatorio,Alto Magdalena,Cundinamarca,GIRARDOT,URBANO,Emitir por encima de los parámetros establecid...,Sin definir,Sin definir,11.736503,-41.043145


**6.** Estandarizar los tipos de fuente

In [61]:
df["TipoFuenteEmision"].value_counts()

TipoFuenteEmision
Sin definir                            326
Horno                                  131
Caldera                                 24
Caldera horno                           14
Secadores                                7
Planta de asfalto                        6
Molino                                   3
Chimenea 1                               3
Trituradora                              2
Noaplica (área de operación)             2
Reactor                                  1
Planta de asfalto adm                    1
Barrilado de grafito                     1
Filtro molino pendular                   1
Aspiración molino danioni i              1
Triturador de escombros                  1
Campana de extracción  plomo 1           1
Horno de secado                          1
Horno arcillas de soacha tipo túnel      1
Triturador de material                   1
Horno túnel 1 soacha 2                   1
Chimenea triunfo central                 1
700 bhp vr2                         

In [62]:
df["TipoFuenteEmision"] = df["TipoFuenteEmision"].replace({
    "Chimenea 1": "Chimenea",
    "Noaplica (área de operación)": "No aplica (área de operación)",
    "Planta de asfalto adm": "Planta de asfalto",
    "Barrilado de grafito": "Horno",
    "Filtro molino pendular": "Molino",
    "Aspiración molino danioni i": "Molino",
    "Triturador de escombros": "Trituradora",
    "Campana de extracción  plomo 1": "Horno",
    "Horno de secado": "Horno",
    "Horno arcillas de soacha tipo túnel": "Horno",
    "Triturador de material": "Trituradora",
    "Horno túnel 1 soacha 2": "Horno",
    "Chimenea triunfo central": "Chimenea",
    "700 bhp vr2": "Caldera",
    "Planta de mezcla asfáltica": "Planta de asfalto",
    "Batería de coquización a": "Batería de coquización",
    "Molino buhler": "Molino",
    "Planta trituradora": "Trituradora",
    "Batería de producción de coque": "Batería de coquización",
})

In [63]:
df["TipoFuenteEmision"].value_counts()

TipoFuenteEmision
Sin definir                      326
Horno                            136
Caldera                           25
Caldera horno                     14
Planta de asfalto                  8
Secadores                          7
Molino                             6
Trituradora                        5
Chimenea                           4
No aplica (área de operación)      2
Batería de coquización             2
Reactor                            1
Name: count, dtype: int64